In [1]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# Load + filter restaurant/English subset
df = pd.read_parquet("/content/drive/MyDrive/MABSA_Shared_Project/data/master_dataset.parquet")
df["domain"] = df["domain"].str.strip().str.lower()
df["language"] = df["language"].str.strip().str.lower()
restaurant_en = df[(df.domain == "restaurant") & (df.language == "en")].copy()
print("restaurant_en shape:", restaurant_en.shape)

# Label maps
sentiments = ["positive", "negative", "neutral"]
restaurant_categories = sorted(restaurant_en["category"].dropna().unique())

tasd_labels = ["O"]
for cat in restaurant_categories:
    for sent in sentiments:
        tasd_labels.append(f"B-{cat}-{sent}")
        tasd_labels.append(f"I-{cat}-{sent}")
tasd_label2id = {l: i for i, l in enumerate(tasd_labels)}

uabsa_labels = ["O"] + [f"B-{s}" for s in sentiments] + [f"I-{s}" for s in sentiments]
uabsa_label2id = {l: i for i, l in enumerate(uabsa_labels)}

print(f"TASD labels: {len(tasd_labels)} | UABSA labels: {len(uabsa_labels)}")

SPLIT_VAL = "dev"  # dataset uses train/dev/test, not train/validation/test

Mounted at /content/drive
GPU available: True
Device: Tesla T4
restaurant_en shape: (3295, 8)
TASD labels: 79 | UABSA labels: 7


In [ ]:

!pip install evaluate seqeval -q

import numpy as np, evaluate, json, os
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

seqeval = evaluate.load("seqeval")

# BIO tagging (handles both true NaN and literal "NULL" string as missing aspect)
def make_bio_tags(sentence, aspect, category, sentiment, task="tasd"):
    tokens = sentence.split()
    tags = ["O"] * len(tokens)
    if pd.isna(aspect) or str(aspect).strip() == "" or str(aspect).strip().upper() == "NULL":
        return tokens, tags
    aspect_tokens = aspect.split()
    n = len(aspect_tokens)
    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == aspect_tokens:
            if task == "tasd":
                tags[i] = f"B-{category}-{sentiment}"
                for j in range(1, n):
                    tags[i+j] = f"I-{category}-{sentiment}"
            else:
                tags[i] = f"B-{sentiment}"
                for j in range(1, n):
                    tags[i+j] = f"I-{sentiment}"
            break
    return tokens, tags

def build_tagged_df(source_df, task):
    records, original_index = [], []
    for idx, row in source_df.iterrows():
        tokens, tags = make_bio_tags(row.sentence, row.aspect, row.category, row.sentiment, task=task)
        records.append({"tokens": tokens, "tags": tags, "split": row.split})
        original_index.append(idx)
    return pd.DataFrame(records, index=original_index)

tasd_df = build_tagged_df(restaurant_en, task="tasd")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def make_tokenize_fn(label2id_local):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local[tags[word_idx]])
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

tasd_tok_fn = make_tokenize_fn(tasd_label2id)

def prep_dataset(tagged_df, tok_fn):
    train_ds = Dataset.from_pandas(tagged_df[tagged_df.split == "train"][["tokens", "tags"]]).map(tok_fn, batched=True)
    val_ds = Dataset.from_pandas(tagged_df[tagged_df.split == SPLIT_VAL][["tokens", "tags"]]).map(tok_fn, batched=True)
    return train_ds, val_ds

tasd_train_ds, tasd_val_ds = prep_dataset(tasd_df, tasd_tok_fn)
print("TASD train/val:", len(tasd_train_ds), len(tasd_val_ds))

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

def train_xlmr(train_ds, val_ds, label2id_local, label_list_local, out_dir, seed=42, epochs=15):
    torch.manual_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=len(label2id_local))
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch", save_strategy="no",
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=epochs, weight_decay=0.01, logging_steps=50,
        load_best_model_at_end=False,
        fp16=True, seed=seed,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=make_compute_metrics(label_list_local),
    )
    trainer.train()
    return trainer

def save_checkpoint(trainer, tokenizer, label2id_local, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "label_map.json"), "w") as f:
        json.dump(label2id_local, f)
    check = AutoModelForTokenClassification.from_pretrained(out_dir)
    ok = check.config.num_labels == len(label2id_local)
    print(f"Saved + verified {out_dir}: {ok}")
    return ok

# --- RUN TRAINING ---
tasd_trainer = train_xlmr(tasd_train_ds, tasd_val_ds, tasd_label2id, tasd_labels,
                           "/content/xlmr_tasd_out", seed=42, epochs=15)

# Save ONLY the final checkpoint — this one IS needed later (Phase 4.1)
save_checkpoint(tasd_trainer, tokenizer, tasd_label2id,
    "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_tasd_seed42_final")

tasd_eval = tasd_trainer.evaluate()
print("\n=== XLM-R TASD RESULTS (15 epochs) ===")
print(f"F1: {tasd_eval['eval_f1']:.4f}")
print(f"Precision: {tasd_eval['eval_precision']:.4f}")
print(f"Recall: {tasd_eval['eval_recall']:.4f}")

Map:   0%|          | 0/1989 [00:00<?, ? examples/s]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

TASD train/val: 1989 507


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved + verified /content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_tasd_seed42_final: True


Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.112414,0.255502,15,0.407125,0.416667,0.398010



=== XLM-R TASD RESULTS (15 epochs) ===
F1: 0.4071
Precision: 0.4167
Recall: 0.3980


In [ ]:

import numpy as np

def bio_to_triples(tokens, tags):
    """Convert a BIO tag sequence back into a set of (aspect_text, category, sentiment) triples."""
    triples = set()
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag.startswith("B-"):
            _, category, sentiment = tag.split("-", 2)
            aspect_tokens = [tokens[i]]
            j = i + 1
            while j < len(tags) and tags[j] == f"I-{category}-{sentiment}":
                aspect_tokens.append(tokens[j])
                j += 1
            aspect_text = " ".join(aspect_tokens).lower()
            triples.add((aspect_text, category.lower(), sentiment.lower()))
            i = j
        else:
            i += 1
    return triples

def score_xlmr_tasd_as_triples(trainer, val_ds, tagged_df, label_list_local):
    predictions = trainer.predict(val_ds)
    preds = np.argmax(predictions.predictions, axis=2)
    label_ids = predictions.label_ids

    # Need original tokens (not sub-word-aligned) — pull from tagged_df in the same order as val_ds
    val_tokens_list = tagged_df[tagged_df.split == SPLIT_VAL]["tokens"].tolist()

    tp, fp, fn = 0, 0, 0
    exact_matches = 0
    n = len(val_tokens_list)

    for tokens, pred_row, label_row in zip(val_tokens_list, preds, label_ids):
        pred_tags, gold_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                pred_tags.append(label_list_local[p])
                gold_tags.append(label_list_local[l])

        min_len = min(len(tokens), len(pred_tags))
        pred_set = bio_to_triples(tokens[:min_len], pred_tags[:min_len])
        gold_set = bio_to_triples(tokens[:min_len], gold_tags[:min_len])

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        if pred_set == gold_set:
            exact_matches += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    triple_level_accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    sentence_level_accuracy = exact_matches / n if n > 0 else 0.0

    return {
        "f1": f1, "precision": precision, "recall": recall,
        "triple_level_accuracy": triple_level_accuracy,
        "sentence_level_accuracy": sentence_level_accuracy,
        "tp": tp, "fp": fp, "fn": fn,
        "n_sentences": n, "n_exact_matches": exact_matches,
    }

scores = score_xlmr_tasd_as_triples(tasd_trainer, tasd_val_ds, tasd_df, tasd_labels)

print("\n=== XLM-R TASD — TRIPLE-BASED SCORES (comparable to mT5) ===")
print(f"F1:                        {scores['f1']:.4f}")
print(f"Precision:                 {scores['precision']:.4f}")
print(f"Recall:                    {scores['recall']:.4f}")
print(f"Triple-level Accuracy:     {scores['triple_level_accuracy']:.4f}  (TP / (TP+FP+FN))")
print(f"Sentence-level Accuracy:   {scores['sentence_level_accuracy']:.4f}  ({scores['n_exact_matches']}/{scores['n_sentences']} exact set matches)")


=== XLM-R TASD — TRIPLE-BASED SCORES (comparable to mT5) ===
F1:                        0.4274
Precision:                 0.4551
Recall:                    0.4030
Triple-level Accuracy:     0.2718  (TP / (TP+FP+FN))
Sentence-level Accuracy:   0.4477  (227/507 exact set matches)


In [ ]:
!pip install evaluate seqeval -q

import numpy as np, evaluate, json, os
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

seqeval = evaluate.load("seqeval")

def make_bio_tags(sentence, aspect, category, sentiment, task="uabsa"):
    tokens = sentence.split()
    tags = ["O"] * len(tokens)
    if pd.isna(aspect) or str(aspect).strip() == "" or str(aspect).strip().upper() == "NULL":
        return tokens, tags
    aspect_tokens = aspect.split()
    n = len(aspect_tokens)
    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == aspect_tokens:
            tags[i] = f"B-{sentiment}"
            for j in range(1, n):
                tags[i+j] = f"I-{sentiment}"
            break
    return tokens, tags

def build_tagged_df(source_df, task):
    records, original_index = [], []
    for idx, row in source_df.iterrows():
        tokens, tags = make_bio_tags(row.sentence, row.aspect, row.category, row.sentiment, task=task)
        records.append({"tokens": tokens, "tags": tags, "split": row.split})
        original_index.append(idx)
    return pd.DataFrame(records, index=original_index)

uabsa_df = build_tagged_df(restaurant_en, task="uabsa")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def make_tokenize_fn(label2id_local):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local[tags[word_idx]])
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

sentiments = ["positive", "negative", "neutral"]
uabsa_labels = ["O"] + [f"B-{s}" for s in sentiments] + [f"I-{s}" for s in sentiments]
uabsa_label2id = {l: i for i, l in enumerate(uabsa_labels)}

uabsa_tok_fn = make_tokenize_fn(uabsa_label2id)

def prep_dataset(tagged_df, tok_fn):
    train_ds = Dataset.from_pandas(tagged_df[tagged_df.split == "train"][["tokens", "tags"]]).map(tok_fn, batched=True)
    val_ds = Dataset.from_pandas(tagged_df[tagged_df.split == SPLIT_VAL][["tokens", "tags"]]).map(tok_fn, batched=True)
    return train_ds, val_ds

uabsa_train_ds, uabsa_val_ds = prep_dataset(uabsa_df, uabsa_tok_fn)
print("UABSA train/val:", len(uabsa_train_ds), len(uabsa_val_ds))

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

def train_xlmr(train_ds, val_ds, label2id_local, label_list_local, out_dir, seed=42, epochs=15):
    torch.manual_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=len(label2id_local))
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch", save_strategy="no",   # avoids disk-full crash
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=epochs, weight_decay=0.01, logging_steps=50,
        load_best_model_at_end=False,
        fp16=True, seed=seed,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=make_compute_metrics(label_list_local),
    )
    trainer.train()
    return trainer

def save_checkpoint(trainer, tokenizer, label2id_local, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "label_map.json"), "w") as f:
        json.dump(label2id_local, f)
    check = AutoModelForTokenClassification.from_pretrained(out_dir)
    ok = check.config.num_labels == len(label2id_local)
    print(f"Saved + verified {out_dir}: {ok}")
    return ok

uabsa_trainer = train_xlmr(uabsa_train_ds, uabsa_val_ds, uabsa_label2id, uabsa_labels,
                            "/content/xlmr_uabsa_out", seed=42, epochs=15)

save_checkpoint(uabsa_trainer, tokenizer, uabsa_label2id,
    "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_uabsa_seed42_final")

uabsa_eval = uabsa_trainer.evaluate()
print("\n=== XLM-R UABSA RESULTS (15 epochs) ===")
print(f"F1: {uabsa_eval['eval_f1']:.4f}")
print(f"Precision: {uabsa_eval['eval_precision']:.4f}")
print(f"Recall: {uabsa_eval['eval_recall']:.4f}")

Map:   0%|          | 0/1989 [00:00<?, ? examples/s]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

UABSA train/val: 1989 507


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved + verified /content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_uabsa_seed42_final: True


Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.063935,0.162221,15,0.524374,0.557423,0.495025



=== XLM-R UABSA RESULTS (15 epochs) ===
F1: 0.5244
Precision: 0.5574
Recall: 0.4950


In [ ]:
import numpy as np

def score_xlmr_token_level(trainer, val_ds, label_list_local):
    predictions = trainer.predict(val_ds)
    preds = np.argmax(predictions.predictions, axis=2)
    label_ids = predictions.label_ids

    true_labels = [[label_list_local[l] for l in label if l != -100] for label in label_ids]
    true_preds = [[label_list_local[p] for (p, l) in zip(pred, label) if l != -100]
                  for pred, label in zip(preds, label_ids)]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "f1": results["overall_f1"],
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "accuracy": results["overall_accuracy"],
        "per_entity": {k: v for k, v in results.items()
                        if k not in ("overall_f1", "overall_precision", "overall_recall", "overall_accuracy")}
    }

token_scores = score_xlmr_token_level(uabsa_trainer, uabsa_val_ds, uabsa_labels)

print("=== XLM-R UABSA — TOKEN-LEVEL SCORES ===")
print(f"F1:         {token_scores['f1']:.4f}")
print(f"Precision:  {token_scores['precision']:.4f}")
print(f"Recall:     {token_scores['recall']:.4f}")
print(f"Accuracy:   {token_scores['accuracy']:.4f}  (token-level tagging accuracy)")
print("\n--- Per-sentiment breakdown ---")
for sentiment, metrics in token_scores["per_entity"].items():
    if isinstance(metrics, dict):
        print(f"{sentiment}: P={metrics['precision']:.4f}, R={metrics['recall']:.4f}, "
              f"F1={metrics['f1']:.4f}, support={metrics['number']}")


def bio_to_pairs(tokens, tags):
    """Convert BIO tags into a set of (aspect_text, sentiment) pairs (no category for UABSA)."""
    pairs = set()
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag.startswith("B-"):
            sentiment = tag.split("-", 1)[1]
            aspect_tokens = [tokens[i]]
            j = i + 1
            while j < len(tags) and tags[j] == f"I-{sentiment}":
                aspect_tokens.append(tokens[j])
                j += 1
            aspect_text = " ".join(aspect_tokens).lower()
            pairs.add((aspect_text, sentiment.lower()))
            i = j
        else:
            i += 1
    return pairs

def score_xlmr_uabsa_as_pairs(trainer, val_ds, tagged_df, label_list_local):
    predictions = trainer.predict(val_ds)
    preds = np.argmax(predictions.predictions, axis=2)
    label_ids = predictions.label_ids

    val_tokens_list = tagged_df[tagged_df.split == SPLIT_VAL]["tokens"].tolist()

    tp, fp, fn = 0, 0, 0
    exact_matches = 0
    n = len(val_tokens_list)

    for tokens, pred_row, label_row in zip(val_tokens_list, preds, label_ids):
        pred_tags, gold_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                pred_tags.append(label_list_local[p])
                gold_tags.append(label_list_local[l])

        min_len = min(len(tokens), len(pred_tags))
        pred_set = bio_to_pairs(tokens[:min_len], pred_tags[:min_len])
        gold_set = bio_to_pairs(tokens[:min_len], gold_tags[:min_len])

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        if pred_set == gold_set:
            exact_matches += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    triple_level_accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    sentence_level_accuracy = exact_matches / n if n > 0 else 0.0

    return {
        "f1": f1, "precision": precision, "recall": recall,
        "triple_level_accuracy": triple_level_accuracy,
        "sentence_level_accuracy": sentence_level_accuracy,
        "tp": tp, "fp": fp, "fn": fn,
        "n_sentences": n, "n_exact_matches": exact_matches,
    }

pair_scores = score_xlmr_uabsa_as_pairs(uabsa_trainer, uabsa_val_ds, uabsa_df, uabsa_labels)

print("\n=== XLM-R UABSA — PAIR-BASED SCORES (comparable to mT5) ===")
print(f"F1:                        {pair_scores['f1']:.4f}")
print(f"Precision:                 {pair_scores['precision']:.4f}")
print(f"Recall:                    {pair_scores['recall']:.4f}")
print(f"Triple-level Accuracy:     {pair_scores['triple_level_accuracy']:.4f}  (TP / (TP+FP+FN))")
print(f"Sentence-level Accuracy:   {pair_scores['sentence_level_accuracy']:.4f}  "
      f"({pair_scores['n_exact_matches']}/{pair_scores['n_sentences']} exact set matches)")

=== XLM-R UABSA — TOKEN-LEVEL SCORES ===
F1:         0.5244
Precision:  0.5574
Recall:     0.4950
Accuracy:   0.9472  (token-level tagging accuracy)

--- Per-sentiment breakdown ---
negative: P=0.4519, R=0.4747, F1=0.4631, support=99
neutral: P=0.2500, R=0.1000, F1=0.1429, support=20
positive: P=0.6122, R=0.5300, F1=0.5682, support=283



=== XLM-R UABSA — PAIR-BASED SCORES (comparable to mT5) ===
F1:                        0.5364
Precision:                 0.5853
Recall:                    0.4950
Triple-level Accuracy:     0.3665  (TP / (TP+FP+FN))
Sentence-level Accuracy:   0.5030  (255/507 exact set matches)


In [ ]:
import numpy as np
from transformers import (AutoTokenizer, MT5ForConditionalGeneration,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq)
from datasets import Dataset

def build_seq2seq_df(source_df, task="tasd"):
    records = []
    for (sentence, split), group in source_df.groupby(["sentence", "split"]):
        targets = []
        for _, row in group.iterrows():
            aspect = str(row.aspect).strip()
            if aspect.upper() == "NULL" or aspect == "" or pd.isna(row.aspect):
                continue
            if task == "tasd":
                targets.append(f"{aspect} | {row.category} | {row.sentiment}")
            else:
                targets.append(f"{aspect} | {row.sentiment}")
        target_text = " ; ".join(targets) if targets else "none"
        records.append({"sentence": sentence, "split": split, "target": target_text})
    return pd.DataFrame(records)

tasd_seq2seq_df = build_seq2seq_df(restaurant_en, task="tasd")

mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")

def make_mt5_tokenize_fn(prefix="Extract aspects and sentiments: "):
    def tokenize_fn(examples):
        inputs = [prefix + s for s in examples["sentence"]]
        model_inputs = mt5_tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
        labels = mt5_tokenizer(text_target=examples["target"], max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize_fn

tasd_mt5_tok_fn = make_mt5_tokenize_fn()

def prep_mt5_dataset(seq2seq_df, tok_fn):
    train_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == "train"][["sentence", "target"]]).map(tok_fn, batched=True)
    val_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == SPLIT_VAL][["sentence", "target"]]).map(tok_fn, batched=True)
    return train_ds, val_ds

tasd_mt5_train, tasd_mt5_val = prep_mt5_dataset(tasd_seq2seq_df, tasd_mt5_tok_fn)
print("TASD mT5 train/val:", len(tasd_mt5_train), len(tasd_mt5_val))

mt5_data_collator = DataCollatorForSeq2Seq(tokenizer=mt5_tokenizer, model=None)

def train_mt5(train_ds, val_ds, out_dir, epochs=15, seed=42):
    torch.manual_seed(seed)
    model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch", save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=2, per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs, weight_decay=0.01, logging_steps=50,
        predict_with_generate=True, generation_max_length=128,
        bf16=True, gradient_checkpointing=True,
        seed=seed,
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=mt5_tokenizer, data_collator=mt5_data_collator,
    )
    trainer.train()
    return trainer

def parse_tasd_output(text):
    text = text.strip()
    if text.lower() == "none" or text == "":
        return set()
    triples = set()
    for chunk in text.split(";"):
        parts = [p.strip() for p in chunk.split("|")]
        if len(parts) == 3:
            triples.add((parts[0].lower(), parts[1].lower(), parts[2].lower()))
    return triples

def evaluate_mt5_tasd(trainer, val_ds, tokenizer, num_beams=4):
    predictions = trainer.predict(val_ds, num_beams=num_beams, max_length=128)
    pred_ids = predictions.predictions
    label_ids = predictions.label_ids

    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids).astype(np.int64)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id).astype(np.int64)
    vocab_size = tokenizer.vocab_size + len(tokenizer.get_added_vocab())
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    label_ids = np.clip(label_ids, 0, vocab_size - 1)

    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    tp, fp, fn = 0, 0, 0
    for pred_text, gold_text in zip(pred_texts, gold_texts):
        pred_set = parse_tasd_output(pred_text)
        gold_set = parse_tasd_output(gold_text)
        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn,
            "sample_preds": list(zip(pred_texts[:8], gold_texts[:8]))}

tasd_mt5_trainer = train_mt5(tasd_mt5_train, tasd_mt5_val, "/content/mt5_tasd_out", epochs=15)

tasd_mt5_results = evaluate_mt5_tasd(tasd_mt5_trainer, tasd_mt5_val, mt5_tokenizer, num_beams=4)

print("\n=== mT5 TASD RESULTS (15 epochs) ===")
print(f"F1: {tasd_mt5_results['f1']:.4f}")
print(f"Precision: {tasd_mt5_results['precision']:.4f}")
print(f"Recall: {tasd_mt5_results['recall']:.4f}")

Map:   0%|          | 0/1263 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

TASD mT5 train/val: 1263 316


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss
1,6.833470,0.778529
2,2.111808,0.529725
3,1.516106,0.296525
4,0.955255,0.176846
5,0.498418,0.059619
6,0.243648,0.049012
7,0.152989,0.041676
8,0.112749,0.038183
9,0.100931,0.037292
10,0.084621,0.037032



=== mT5 TASD RESULTS (15 epochs) ===
F1: 0.6169
Precision: 0.6177
Recall: 0.6160


In [ ]:
import numpy as np

def score_mt5_tasd(trainer, val_ds, tokenizer, num_beams=4):
    predictions = trainer.predict(val_ds, num_beams=num_beams, max_length=128)
    pred_ids = predictions.predictions
    label_ids = predictions.label_ids

    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids).astype(np.int64)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id).astype(np.int64)
    vocab_size = tokenizer.vocab_size + len(tokenizer.get_added_vocab())
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    label_ids = np.clip(label_ids, 0, vocab_size - 1)

    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    tp, fp, fn = 0, 0, 0
    exact_matches = 0
    n = len(pred_texts)

    for pred_text, gold_text in zip(pred_texts, gold_texts):
        pred_set = parse_tasd_output(pred_text)
        gold_set = parse_tasd_output(gold_text)

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        # Sentence-level exact match: predicted triple-set == gold triple-set
        if pred_set == gold_set:
            exact_matches += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    triple_level_accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    sentence_level_accuracy = exact_matches / n if n > 0 else 0.0

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "triple_level_accuracy": triple_level_accuracy,
        "sentence_level_accuracy": sentence_level_accuracy,
        "tp": tp, "fp": fp, "fn": fn,
        "n_sentences": n,
        "n_exact_matches": exact_matches,
    }

scores = score_mt5_tasd(tasd_mt5_trainer, tasd_mt5_val, mt5_tokenizer, num_beams=4)

print("\n=== mT5 TASD — FULL SCORES ===")
print(f"F1:                        {scores['f1']:.4f}")
print(f"Precision:                 {scores['precision']:.4f}")
print(f"Recall:                    {scores['recall']:.4f}")
print(f"Triple-level Accuracy:     {scores['triple_level_accuracy']:.4f}  (TP / (TP+FP+FN))")
print(f"Sentence-level Accuracy:   {scores['sentence_level_accuracy']:.4f}  ({scores['n_exact_matches']}/{scores['n_sentences']} exact set matches)")
print(f"TP={scores['tp']}, FP={scores['fp']}, FN={scores['fn']}")


=== mT5 TASD — FULL SCORES ===
F1:                        0.6169
Precision:                 0.6177
Recall:                    0.6160
Triple-level Accuracy:     0.4460  (TP / (TP+FP+FN))
Sentence-level Accuracy:   0.5506  (174/316 exact set matches)
TP=223, FP=138, FN=139


In [2]:
import numpy as np
from transformers import (AutoTokenizer, MT5ForConditionalGeneration,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq)
from datasets import Dataset

def build_seq2seq_df(source_df, task="uabsa"):
    records = []
    for (sentence, split), group in source_df.groupby(["sentence", "split"]):
        targets = []
        for _, row in group.iterrows():
            aspect = str(row.aspect).strip()
            if aspect.upper() == "NULL" or aspect == "" or pd.isna(row.aspect):
                continue
            if task == "tasd":
                targets.append(f"{aspect} | {row.category} | {row.sentiment}")
            else:
                targets.append(f"{aspect} | {row.sentiment}")
        target_text = " ; ".join(targets) if targets else "none"
        records.append({"sentence": sentence, "split": split, "target": target_text})
    return pd.DataFrame(records)

uabsa_seq2seq_df = build_seq2seq_df(restaurant_en, task="uabsa")

mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")

def make_mt5_tokenize_fn(prefix="Extract aspects and sentiments: "):
    def tokenize_fn(examples):
        inputs = [prefix + s for s in examples["sentence"]]
        model_inputs = mt5_tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
        labels = mt5_tokenizer(text_target=examples["target"], max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize_fn

uabsa_mt5_tok_fn = make_mt5_tokenize_fn()

def prep_mt5_dataset(seq2seq_df, tok_fn):
    train_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == "train"][["sentence", "target"]]).map(tok_fn, batched=True)
    val_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == SPLIT_VAL][["sentence", "target"]]).map(tok_fn, batched=True)
    return train_ds, val_ds

uabsa_mt5_train, uabsa_mt5_val = prep_mt5_dataset(uabsa_seq2seq_df, uabsa_mt5_tok_fn)
print("UABSA mT5 train/val:", len(uabsa_mt5_train), len(uabsa_mt5_val))

mt5_data_collator = DataCollatorForSeq2Seq(tokenizer=mt5_tokenizer, model=None)

def train_mt5(train_ds, val_ds, out_dir, epochs=15, seed = 42):
    torch.manual_seed(seed)
    model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch", save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=2, per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs, weight_decay=0.01, logging_steps=50,
        predict_with_generate=True, generation_max_length=128,
        bf16=True, gradient_checkpointing=True,
        seed = seed,
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=mt5_tokenizer, data_collator=mt5_data_collator,
    )
    trainer.train()
    return trainer

def parse_uabsa_output(text):
    text = text.strip()
    if text.lower() == "none" or text == "":
        return set()
    pairs = set()
    for chunk in text.split(";"):
        parts = [p.strip() for p in chunk.split("|")]
        if len(parts) == 2:
            pairs.add((parts[0].lower(), parts[1].lower()))
    return pairs

def evaluate_mt5_uabsa(trainer, val_ds, tokenizer, num_beams=4):
    predictions = trainer.predict(val_ds, num_beams=num_beams, max_length=128)
    pred_ids = predictions.predictions
    label_ids = predictions.label_ids

    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids).astype(np.int64)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id).astype(np.int64)
    vocab_size = tokenizer.vocab_size + len(tokenizer.get_added_vocab())
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    label_ids = np.clip(label_ids, 0, vocab_size - 1)

    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    tp, fp, fn = 0, 0, 0
    for pred_text, gold_text in zip(pred_texts, gold_texts):
        pred_set = parse_uabsa_output(pred_text)
        gold_set = parse_uabsa_output(gold_text)
        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn,
            "sample_preds": list(zip(pred_texts[:8], gold_texts[:8]))}

# --- RUN TRAINING + EVAL (no model save — not needed downstream) ---
uabsa_mt5_trainer = train_mt5(uabsa_mt5_train, uabsa_mt5_val, "/content/mt5_uabsa_out", epochs=15)

uabsa_mt5_results = evaluate_mt5_uabsa(uabsa_mt5_trainer, uabsa_mt5_val, mt5_tokenizer, num_beams=4)

print("\n=== mT5 UABSA RESULTS (15 epochs) ===")
print(f"F1: {uabsa_mt5_results['f1']:.4f}")
print(f"Precision: {uabsa_mt5_results['precision']:.4f}")
print(f"Recall: {uabsa_mt5_results['recall']:.4f}")

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/1263 [00:00<?, ? examples/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,3.394024,0.404159
2,2.272488,0.278961
3,1.070546,0.226961
4,1.454724,0.086610
5,0.292870,0.051393
6,0.193947,0.045785
7,0.125859,0.036490
8,0.091472,0.033640
9,0.080666,0.030628
10,0.060519,0.029615



=== mT5 UABSA RESULTS (15 epochs) ===
F1: 0.6959
Precision: 0.6959
Recall: 0.6959
